# Plotting a MuData with scanpy's HoloViews functions

HoloViews can only read an `AnnData` (through `hv-anndata`). A `MuData` is not an `AnnData`, so `hv.Scatter(mdata, ...)`
used to fall back to HoloViews' dictionary interface and never finish.

This branch adds `MuDataInterface`, a small subclass of `hv_anndata.AnnDataInterface` (`src/muon/_hv.py`), and registers it when
scanpy's preset is `ScanpyV2Preview` at the time `muon` is imported. scanpy's HoloViews plotting functions
(`sc.pl.scatter`, `sc.pl.umap`, `sc.pl.pca`, ...) are thin wrappers around `hv.Scatter(adata, ...)`, so they accept a `MuData` as soon as the interface exists.

Muon's own `mu.pl.scatter`, `mu.pl.embedding`, `mu.pl.umap` and `mu.pl.mofa` follow the same preset, like scanpy's functions: matplotlib with the default preset,
HoloViews with `ScanpyV2Preview`. Before this change they failed under that preset, because they called `sc.pl.embedding` (which does not exist there) and `sc.pl.scatter(x=..., y=...)`
(whose signature changed).

**Order matters:** set the preset *before* importing `muon`.

**Requirements:** `mudata>=0.4`, `anndata>=0.13`, `scanpy[scanpy2]`, `hv-anndata`, `holoviews`, `bokeh`.

```bash
git clone https://github.com/ellathomas01/muon.git && cd muon
git checkout mudata-holoviews-interface
uv venv --python 3.12 && source .venv/bin/activate
uv pip install --prerelease=allow "scanpy[scanpy2]" hv-anndata holoviews bokeh panel jupyterlab
uv pip install -U mudata
uv pip install -e .
```

## 0. Setup: the preset comes first

In [ ]:
from importlib.metadata import version

import scanpy as sc

sc.settings.preset = sc.Preset.ScanpyV2Preview  # must happen before `import muon`

import anndata as ad  # noqa: E402
import holoviews as hv  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

import muon as mu  # noqa: E402
from muon import _hv  # noqa: E402

A = sc.pl.hv_init("bokeh")  # scanpy's own initialiser; returns the accessor `A`

for pkg in ("muon", "mudata", "anndata", "scanpy", "holoviews", "hv-anndata", "bokeh"):
    print(f"{pkg:<11}", version(pkg))
print()
print("MuData interface registered:", "mudata" in hv.core.data.datatypes, "| in muon:", _hv._registered)

## 1. Test data

`rna` is scanpy's `pbmc68k_reduced`; `prot` is a made-up protein modality built from three marker genes plus noise.
Under the v2 preset scanpy stores embeddings without the `X_` prefix (`obsm["pca"]`, `obsm["umap"]`), and `sc.pl.umap` looks in `obsm["umap"]`.
The MuData gets a joint embedding in `obsm["X_joint_pca"]` and the RNA UMAP copied to `obsm["umap"]`.

In [ ]:
rna = sc.datasets.pbmc68k_reduced()
markers = ["CD79A", "CST3", "LYZ"]
rng = np.random.default_rng(0)
values = np.asarray(rna[:, markers].X) + rng.normal(0, 0.3, (rna.n_obs, len(markers)))
prot = ad.AnnData(X=values.astype("float32"))
prot.obs_names = rna.obs_names
prot.var_names = [f"prot_{m}" for m in markers]

mdata = mu.MuData({"rna": rna, "prot": prot})
mdata.pull_obs(["bulk_labels", "S_score"])
mdata.obs["condition"] = pd.Categorical(mdata.obs["rna:bulk_labels"].astype(str))
joint = np.hstack([rna.obsm["pca"][:, :10], prot.X])
mdata.obsm["X_joint_pca"] = PCA(n_components=2, random_state=0).fit_transform(joint)
mdata.obsm["umap"] = rna.obsm["umap"]
mdata

## 2. `sc.pl.umap` on a MuData

In [ ]:
sc.pl.umap(mdata, color=A.obs["condition"]).opts(title="UMAP by condition", tools=["hover"])

## 3. `sc.pl.scatter` on a MuData: a joint embedding, then several colours

scanpy's v2 plots have no hover tool by default, so it is added with `.opts(tools=["hover"])`. For a `Layout`, options go on the plots (`hv.opts.Scatter(...)`).

In [ ]:
sc.pl.scatter(mdata, A.obsm["X_joint_pca"][:, [0, 1]], color=A.obs["condition"]).opts(tools=["hover"])

In [ ]:
sc.pl.scatter(mdata, A.obsm["X_joint_pca"][:, [0, 1]], color=A.obs[["rna:S_score", "condition"]]).opts(
    hv.opts.Scatter(tools=["hover"])
)

## 4. Plain HoloViews on a MuData

In [ ]:
hv.Scatter(mdata, A.obsm["umap"][0], [A.obsm["umap"][1], A.obs["rna:S_score"]]).opts(
    backend="bokeh", color=A.obs["rna:S_score"], colorbar=True, cmap="viridis", aspect="square", tools=["hover"]
)

## 5. Muon's own plotting functions

With the v2 preset, `mu.pl.embedding`, `mu.pl.umap` and `mu.pl.scatter` return HoloViews plots. They still take strings, and find features in any modality
(`"CST3"`, `"prot_CST3"`). One colour gives a single plot, several colours a `Layout`. The plots have a hover tool by default, and extra keyword arguments are HoloViews options.

In [ ]:
mu.pl.embedding(mdata, basis="rna:umap", color="condition").opts(aspect="square")

In [ ]:
mu.pl.embedding(mdata, basis="X_joint_pca", color=["CST3", "prot_CST3"])

In [ ]:
mu.pl.scatter(mdata, x="CST3", y="prot_CST3", color="condition", size=5)

## 6. Checks

Each row compares something computed through the interface with the data in the MuData. Any row that is not `OK` needs attention.

In [ ]:
rows = []


def check(name, ok, detail=""):
    """Record the result of one check."""
    rows.append({"check": name, "verdict": "OK" if ok else "FAIL", "detail": detail})


def attempt(name, fn):
    """Run `fn()` (returning a bool) and record the result, or the error it raised."""
    try:
        check(name, bool(fn()))
    except Exception as e:  # noqa: BLE001
        check(name, False, f"{type(e).__name__}: {str(e)[:100]}")


attempt("interface registered", lambda: "mudata" in hv.core.data.datatypes)

umap_plot = sc.pl.umap(mdata, color=A.obs["condition"])
attempt("sc.pl.umap has one point per cell", lambda: len(umap_plot) == mdata.n_obs)
attempt(
    "sc.pl.umap x equals mdata.obsm['umap'][:, 0]",
    lambda: np.allclose(umap_plot.dimension_values(0), mdata.obsm["umap"][:, 0]),
)
attempt(
    "sc.pl.umap colour equals mdata.obs['condition']",
    lambda: list(umap_plot.dimension_values(2)) == list(mdata.obs["condition"]),
)

scatter_plot = sc.pl.scatter(mdata, A.obsm["X_joint_pca"][:, [0, 1]], color=A.obs["condition"])
attempt(
    "sc.pl.scatter y equals the joint embedding's second column",
    lambda: np.allclose(scatter_plot.dimension_values(1), mdata.obsm["X_joint_pca"][:, 1]),
)

layout = sc.pl.scatter(mdata, A.obsm["X_joint_pca"][:, [0, 1]], color=A.obs[["rna:S_score", "condition"]])
attempt("two colours give a Layout of two plots", lambda: isinstance(layout, hv.Layout) and len(layout) == 2)

view = sc.pl.scatter(mdata[:200], A.obsm["X_joint_pca"][:, [0, 1]], color=A.obs["condition"])
attempt("a view of 200 cells gives 200 points", lambda: len(view) == 200)

label = mdata.obs["condition"].iloc[0]
selected = scatter_plot.select(**{repr(A.obs["condition"]): label})
attempt("select keeps only the chosen group", lambda: len(selected) == int((mdata.obs["condition"] == label).sum()))

attempt("renders with bokeh", lambda: len(hv.render(umap_plot, backend="bokeh").renderers) >= 1)

mu_embedding = mu.pl.embedding(mdata, basis="rna:umap", color="condition")
attempt(
    "mu.pl.embedding returns one Scatter with one point per cell",
    lambda: isinstance(mu_embedding, hv.Scatter) and len(mu_embedding) == mdata.n_obs,
)
attempt(
    "mu.pl.embedding x equals rna.obsm['umap'][:, 0]",
    lambda: np.allclose(mu_embedding.dimension_values(0), rna.obsm["umap"][:, 0]),
)
mu_layout = mu.pl.embedding(mdata, basis="rna:umap", color=["CST3", "prot_CST3"])
attempt(
    "mu.pl.embedding with two colours gives a Layout of two plots",
    lambda: isinstance(mu_layout, hv.Layout) and len(mu_layout) == 2,
)
attempt("mu.pl.umap has one point per cell", lambda: len(mu.pl.umap(mdata, color="condition")) == mdata.n_obs)
mu_scatter = mu.pl.scatter(mdata, x="CST3", y="prot_CST3", color="condition")
cst3 = np.asarray(rna[:, "CST3"].X).ravel()
attempt("mu.pl.scatter x equals the gene CST3 in rna", lambda: np.allclose(mu_scatter.dimension_values(0), cst3))
attempt(
    "mu.pl.scatter y equals the protein prot_CST3",
    lambda: np.allclose(mu_scatter.dimension_values(1), prot[:, "prot_CST3"].X.ravel()),
)

adata_plot = sc.pl.umap(rna, color=A.obs["bulk_labels"])
attempt("a plain AnnData still works", lambda: len(adata_plot) == rna.n_obs)

results = pd.DataFrame(rows)
results

In [ ]:
bad = results[results["verdict"] != "OK"]
print(f"{len(results)} checks, {len(bad)} failed")
assert bad.empty, "see the table above"

## 7. What is and is not covered

- `mu.pl.histogram` and `mu.pl.mofa_loadings` are not covered. They fail with `No module named 'scanpy.plotting._utils'`, with either preset, because current scanpy moved its plotting internals to `scanpy.plotting.legacy`. That is a separate issue.
- Under the v2 preset, `mu.pl.*` keyword arguments are HoloViews options (matplotlib ones such as `show=False` are rejected, as in scanpy's v2 functions). For a plain `AnnData`, only `.obs` columns can be plotted, and `use_raw`, `layer(s)` and `gene_symbols` raise `NotImplementedError`.
- Accessors for a single modality, e.g. `A.mod["rna"].obsm["umap"][0]`, need `mudata.acc.MuAcc`:
  `from mudata.acc import MuAcc; from hv_anndata import AdDim; A = MuAcc(ref_class=AdDim)`.
  The `A` returned by `sc.pl.hv_init` only has the accessors that exist on both `AnnData` and `MuData` (`A.obs`, `A.obsm`, ...).
- The interface is registered at `import muon`, so the preset must be set before that. Setting the preset afterwards has no effect on the registration.